# ECG Image Digitization (Image -> 1D Signal) v1
This notebook trains a Convolutional Autoencoder-style network to extract 1D signals from 2D ECG images.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import os
import glob
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Reshape

## 1. Synthetic Data Generation
We load the 1D signals and convert them to images with simulated background noise.

In [ ]:
# Load data
csv_paths = glob.glob('/kaggle/input/**/ptbdb_normal.csv', recursive=True)
if not csv_paths:
    raise FileNotFoundError('Could not find ptbdb_normal.csv in /kaggle/input/')
print(f'Found dataset at {csv_paths[0]}')
df_normal = pd.read_csv(csv_paths[0], header=None)
signals = df_normal.iloc[:, :-1].values

def generate_ecg_image(signal, img_size=(128, 128)):
    fig, ax = plt.subplots(figsize=(2, 2), dpi=64)
    ax.plot(signal, color='black', linewidth=1.5)
    ax.axis('off')
    fig.canvas.draw()
    
    # Convert to numpy array
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)
    
    # Convert to grayscale and resize
    img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    img_resized = cv2.resize(img_gray, img_size)
    
    # Normalize
    img_normalized = img_resized / 255.0
    return np.expand_dims(img_normalized, axis=-1)

# Generate a subset for training (e.g., 2000 samples for MVP)
NUM_SAMPLES = 2000
X_images = []
Y_signals = []

print('Generating synthetic images...')
for i in tqdm(range(NUM_SAMPLES)):
    sig = signals[i]
    img = generate_ecg_image(sig)
    X_images.append(img)
    Y_signals.append(sig)

X_images = np.array(X_images)
Y_signals = np.array(Y_signals)
print(f'X shape: {X_images.shape}, Y shape: {Y_signals.shape}')

## 2. Model Architecture
A CNN encoder followed by a Dense decoder to output 187 signal points.

In [ ]:
def build_digitizer(input_shape=(128, 128, 1)):
    inputs = Input(shape=input_shape)
    
    x = Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
    x = MaxPooling2D((2,2))(x)
    
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2,2))(x)
    
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2,2))(x)
    
    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.2)(x)
    
    outputs = Dense(187, activation='linear')(x) # 187 points in heartbeat signal
    
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

model = build_digitizer()
model.summary()

## 3. Training

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_images, Y_signals, test_size=0.2, random_state=42)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32
)

## 4. Evaluation & Visualization

In [ ]:
predictions = model.predict(X_val[:5])

for i in range(3):
    plt.figure(figsize=(10, 3))
    plt.plot(y_val[i], label='True Signal', color='blue')
    plt.plot(predictions[i], label='Predicted Signal', color='red', linestyle='dashed')
    plt.legend()
    plt.title(f'Sample {i+1} Reconstruction')
    plt.show()